# Calculate mean benefit and change in admissions if stroke unit gains MT


__Results__:

For each stroke unit, and for including and excluding lossy drip-and-ship transfers, save these files:


## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.geography_processing import Geoprocessing
from classes.model import Model
from classes.scenario import Scenario
from classes.add_net_benefit import add_net_benefit

In [ ]:
dir_output = './output'

## Load data

Scenario parameters:

In [2]:
p = './output/scenario_parameters.csv'

# Use converters to turn units column from strings to lists.
# Assumes that no postcodes contain spaces!
converters = {
    'new_mt_units': lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()),
}
df_scen = pd.read_csv(p, converters=converters)

In [3]:
display(df_scen.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,single_unit_changed,prop_lvo_dripship_treated_with_mt,mt_travel_bias_minutes,prop_lvo
0,base,base,[],0,1.0,0.1,0.3
1,base_lossydripshipmt,base,[],0,0.6,0.1,0.3
2,taunworc,taunworc,"[TA15DA, WR51DD]",0,1.0,0.1,0.3
3,taunworc_lossydripshipmt,taunworc,"[TA15DA, WR51DD]",0,0.6,0.1,0.3
4,only_RG15AN,only_RG15AN,[RG15AN],1,1.0,0.1,0.3
5,only_RG15AN_lossydripshipmt,only_RG15AN,[RG15AN],1,0.6,0.1,0.3


Split into scenarios to calculate in full and scenarios to only calculate as summary:

In [8]:
m = df_scen['single_unit_changed'] == 0
df_scen_full = df_scen.loc[m].drop('single_unit_changed', axis='columns')
df_scen_sing = df_scen.loc[~m].drop('single_unit_changed', axis='columns')

In [10]:
display(df_scen_sing.head(5))

,scenario_name,scenario_geoprocess_name,new_mt_units,prop_lvo_dripship_treated_with_mt,mt_travel_bias_minutes,prop_lvo
4,only_RG15AN,only_RG15AN,[RG15AN],1.0,0.1,0.3
5,only_RG15AN_lossydripshipmt,only_RG15AN,[RG15AN],0.6,0.1,0.3
6,only_HP112TT,only_HP112TT,[HP112TT],1.0,0.1,0.3
7,only_HP112TT_lossydripshipmt,only_HP112TT,[HP112TT],0.6,0.1,0.3
8,only_CH21UL,only_CH21UL,[CH21UL],1.0,0.1,0.3


## Main calculations

In [ ]:
def calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
        options_prop_lvo_dripship_treated_with_mt
    ):
    # Process geographic data (only needed when hospital data changes):
    geo = Geoprocessing(mt_travel_bias=mt_travel_bias,
                        new_mt_centres=new_mt_units)
    geo.run()
    geodata = geo.combined_data.copy().reset_index()
    
    # Set up scenario
    scenario = Scenario({
        'name': 'temporary_name',
        'limit_to_england' : True
    })
    
    # Set up model
    model = Model(
        scenario=scenario,
        geodata=geodata
    )
    
    # Run model
    model.run()
    results = model.full_results

    # Add net benefit column
    results = add_net_benefit(
        results,
        prop_lvo=prop_lvo,
    )
    for p in options_prop_lvo_dripship_treated_with_mt:
        results = add_lossy_net_benefit(
            results, prop_lvo=prop_lvo, prop_lvo_ds_mt=p)

    # Round outcome results:
    cols_mrs = [c for c in results.columns if (
                ('mrs' in c) | ('improvement' in c))]
    cols_util = [c for c in results.columns if ('util' in c)]
    results[cols_mrs] = results[cols_mrs].round(3)
    results[cols_util] = results[cols_util].round(5)
    
    return results

## Run for single-unit changes:

Don't save the full results to file. Gather summary results and save those.

First load in base scenario results for a comparison:

In [ ]:
dict_base_results = {}

for scenario_name in ['base', 'base_lossydripshipmt']:
    p = os.path.join(dir_output, f'results_full_{scenario_name}.csv')
    model_result = pd.read_csv(p)

    # Get benefit for LVO
    base_benefit_by_lsoa = (model_result['lvo_drip_ship_ivt_mt_mrs_0-2'] -
                            model_result['lvo_no_treatment_mrs_0-2'])
    admissions_by_lsoa = model_result['Admissions_2025_Est']

    dict_base_results[scenario_name] = {
        'base_benefit_by_lsoa': base_benefit_by_lsoa,
        'admissions_by_lsoa': admissions_by_lsoa,
        }

In [ ]:
for scen, base_dict in dict_base_results.items():
    # Print mean weighted by admissions
    print(scen)
    b = np.average(dict_base_results['base_benefit_by_lsoa'],
                   weights=dict_base_results['admissions_by_lsoa'])
    print(f"Mean benefit weighted by admissions: {b}")

Run each single-unit scenario:

# TO DO

In [ ]:
hospital_net_benefit = {}
hospital_admissions_benefiting = {}
hospital_benefit_for_those_benefitting = {}

for scen_geo_name, df in df_scen_sing.groupby('scenario_geoprocess_name'):
    # These parameters should be the same for all df in group:
    mt_travel_bias = df.iloc[0]['mt_travel_bias']
    new_mt_units = df.iloc[0]['new_mt_units']
    prop_lvo = df.iloc[0]['prop_lvo']
    # Shared geography and outcome results:
    results_shared = calculate_geo_and_outcomes(mt_travel_bias, new_mt_units)
    # Combine populations with different parameters:
    for i, series_scen in df.iterrows():
        results = results_shared.copy()
        # Name for files:
        scenario_name = series_scen['scenario_name']
        base_name = ('base_lossydripshipmt'
                     if '_lossydripshipmt' in scenario_name else 'base')

        # Add net benefit column
        results = add_net_benefit(
            results,
            prop_lvo=prop_lvo,
            prop_lvo_ds_mt=series_scen['prop_lvo_dripship_treated_with_mt']
        )

        benefit = (results['lvo_drip_ship_ivt_mt_mrs_0-2'] -
                   results['lvo_no_treatment_mrs_0-2'])
        benefit_over_base_by_lsoa = (
            benefit - dict_base_results[base_name]['base_benefit_by_lsoa'])
    
        total_admissions = admissions_by_lsoa.sum()
        hospital_net_benefit[unit] = (
            np.average(benefit_over_base_by_lsoa, weights=admissions_by_lsoa)
            if total_admissions > 0 else np.nan
        )
    
        benefiting_lsoas = benefit_over_base_by_lsoa > 0
        benefiting_benefit = benefit_over_base_by_lsoa.loc[benefiting_lsoas]
        benefiting_admissions = admissions_by_lsoa.loc[benefiting_lsoas]
    
        hospital_admissions_benefiting[unit] = benefiting_admissions.sum()
    
        benefiting_weight = benefiting_admissions.sum()
        hospital_benefit_for_those_benefitting[unit] = (
            np.average(benefiting_benefit, weights=benefiting_admissions)
            if benefiting_weight > 0 else np.nan
        )


results_df = pd.DataFrame({
    'Hospital': list(hospital_net_benefit),
    'Weighted Net Benefit': list(hospital_net_benefit.values()),
    'Admission Catchment Benefiting': list(hospital_admissions_benefiting.values()),
    'Mean Benefit for Those Benefitting': list(
        hospital_benefit_for_those_benefitting.values()
    )
})

results_df = results_df.sort_values(
    by='Weighted Net Benefit',
    ascending=False
)

hospitals = pd.read_csv('data/stroke_hospitals_2025.csv')
results_df = results_df.merge(
    hospitals[['Postcode', 'SSNAP name', 'Hospital_City']],
    how='left',
    left_on='Hospital',
    right_on='Postcode'
).set_index('SSNAP name')

best_benefit = results_df['Weighted Net Benefit'].max()
results_df['Relative Benefit'] = (
    results_df['Weighted Net Benefit'] / best_benefit
    if pd.notna(best_benefit) and best_benefit != 0 else np.nan
).round(3)

clear_output()
results_df.to_csv('output/benefit_of_each_team.csv', index=True)

results_df.head(20)
